# Chapter 3: Assigning Roles (Role Prompting)

**Technique:** Role prompting (persona prompting)

**Contents**
* [Lesson: Why Roles Matter](#lesson)
* [Exercises](#exercises)
* Example Playground

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Why Roles Matter

When you add a role to the system prompt (for example, "You are a meticulous senior code reviewer"), Claude shifts its tone, focus, and depth of analysis to match that persona. This is not a trick; it mirrors how you'd brief a real colleague before asking for feedback.

**Why this matters in practice**

Without a role, Claude answers the question it's given but has no reason to push back, look for edge cases, or assume the worst. With a role, Claude adopts the priorities of that persona:

* A **security auditor** will flag injection vectors you didn't ask about.
* A **senior code reviewer** will notice boundary conditions even in a short snippet.
* An **SRE on call** will focus on operational impact over style.

**Where role prompting pays off most**

* **PR review**: "meticulous senior engineer who rejects off by one errors".
* **Security audit**: "adversarial security researcher looking for injection and auth flaws".
* **Architecture critique**: "staff engineer who always asks about failure modes and scalability".
* **API design review**: "developer experience engineer who focuses on naming and consistency".

**How to structure it**

Put the role in the **system prompt**, not the user turn. System prompt roles persist across the whole conversation. User turn role instructions can be overridden or diluted as the conversation grows.

```js
const SYSTEM_PROMPT = "You are a meticulous senior code reviewer who always checks boundary conditions.";
const response = await getCompletion("Is there a bug in this code?\n<code>...<\/code>", SYSTEM_PROMPT);
```

**The specificity rule**: "You are an expert" rarely helps, because Claude already knows about most topics. Specificity about *what the role cares about* is what changes behavior. "You are a senior reviewer who always checks boundary conditions and rejects functions that silently return undefined" is more useful than "You are a senior reviewer."

In [ ]:
// Same buggy snippet, see how the role changes the quality of feedback.
const BUGGY = `function lastItem(arr) { return arr[arr.length]; }`;

console.log("WITHOUT ROLE:\n", await getCompletion(`Is there a bug in this code?\n${BUGGY}`));
console.log("\nWITH ROLE:\n", await getCompletion(`Is there a bug in this code?\n${BUGGY}`, "You are a meticulous senior code reviewer who always checks boundary conditions."));

## Exercises

### Exercise 3.1: Catch the off by one with a reviewer role

**Scenario**: a teammate opens a PR claiming this utility function is correct:

```js
function lastItem(arr) { return arr[arr.length]; }
```

`arr[arr.length]` is always `undefined`. Arrays are indexed from 0, so the last element is at `arr[arr.length - 1]`. Without a role, Claude may give a lukewarm response. Your job is to assign Claude a role (via `SYSTEM_PROMPT`) that makes it flag the code as incorrect.

**Task**: Replace `SYSTEM_PROMPT` with a role that causes Claude to scrutinize the code and catch the bug. Leave `PROMPT` unchanged.

**Success criteria**: the response contains at least one of: `incorrect`, `not correct`, `bug`, `off-by-one`, `off by one`, `undefined`.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const SYSTEM_PROMPT = "[Replace this text]";
const PROMPT = "A teammate says this returns the last element. Is it correct?\nfunction lastItem(arr) { return arr[arr.length]; }";

const response = await getCompletion(PROMPT, SYSTEM_PROMPT);
const gradeExercise = (text) => includesAny(text, ["incorrect", "not correct", "bug", "off-by-one", "off by one", "undefined"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_3_1_hint } from "../hints.js";
console.log(exercise_3_1_hint);

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Role too generic**: "You are an expert developer" gives Claude a persona but no explicit priorities. A more effective role specifies *what the reviewer cares about*: "You are a senior engineer who rejects any function that can return `undefined` silently." The more targeted the role, the more consistent the output.
* **Role in the user turn instead of system**: placing "Act as a code reviewer" inside the user message works for a single call, but in a conversation with many turns it can drift. System prompt roles are stable across all turns.
* **No domain context**: "senior engineer" is generic. "senior engineer reviewing Node.js services where runtime errors are unacceptable" gives Claude operating constraints, not just a title.

**Stretch challenge**

Extend `SYSTEM_PROMPT` so Claude not only flags the bug but also returns the corrected function. Confirm the response includes `arr.length - 1` or an equivalent fix.

**Reflect**: Which types of code review tasks benefit most from role prompting, and which don't? Consider: stylistic opinions (tabs vs spaces) vs correctness errors (undefined access, race conditions) vs security flaws (injection, unvalidated input). Where would you inject a role prompted Claude call into your team's existing CI/PR pipeline?

## Example Playground

Use the cell below to experiment with different roles on the same prompt. Try assigning an SRE, a security researcher, or an API designer, then observe how the feedback changes.

In [ ]:
// Swap the role and see how the analysis shifts
const SYSTEM_PROMPT = "You are an adversarial security researcher. Assume all inputs are hostile.";
const PROMPT = "Review this Express route handler:\napp.get('/user', (req, res) => { db.query('SELECT * FROM users WHERE id=' + req.query.id, (err, rows) => res.json(rows)); });";
console.log(await getCompletion(PROMPT, SYSTEM_PROMPT));